Ensure Runtime > change runtime type
1. runtype: python3
2. h\w accel: t4 gpy
3. runtime version: 2025.07

In [1]:
!curl -fsSL https://deno.land/install.sh | sh
import os
os.environ['PATH'] += ':/root/.deno/bin'
!deno --version

######################################################################## 100.0%
Archive:  /root/.deno/bin/deno.zip
  inflating: /root/.deno/bin/deno    
Deno was installed successfully to /root/.deno/bin/deno
sh: 105: cannot open /dev/tty: No such device or address
deno 2.9.6 (stable, release, x86_64-unknown-linux-gnu)
v8 15.0.245.2-rusty
typescript 6.0.3


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1. Define Paths
DRIVE_BASE = "/content/drive/MyDrive"
REPO_DIR = f"{DRIVE_BASE}/sutta-tts-model-training"
REPO_SCRIPTS_DIR = f"{REPO_DIR}/scripts"
PIPER_TRAINING = f"{DRIVE_BASE}/piper_training"
# LOCAL_CACHE = "/content/piper_cache"
PIPER_REPO = "/content/piper1-gpl"


In [3]:
# ============================================================
# GIT
# ============================================================
import os
import subprocess


# 2. Clone/Update Repo (Manages scripts & configs)
print(f"📦 Syncing Repo to {REPO_DIR}...")
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dhamma-initiative/sutta-tts-model-training.git {REPO_DIR}
    %cd {REPO_DIR}
    !git switch "colab-trials"
    !git pull
    print("✅ Repo cloned.")
else:
    !cd {REPO_DIR} && git pull
    print("✅ Repo updated.")

# 3. Create Directories
os.makedirs(f"{PIPER_TRAINING}/checkpoints", exist_ok=True)
print("✅ Directories ready.")
%cd /content

📦 Syncing Repo to /content/drive/MyDrive/sutta-tts-model-training...
Already up to date.
✅ Repo updated.
✅ Directories ready.
/content


In [ ]:
# ============================================================
# DOWNLOAD REQUIRED FILES
# ============================================================
# 1. Piper Cache (Pre-computed features)
if not os.path.exists(f"{PIPER_TRAINING}/piper_cache.tgz"):
    print("Fetching piper_cache.tgz...")
    !wget -O "{PIPER_TRAINING}/piper_cache.tgz" "https://drive.usercontent.google.com/download?id=1SKvjCqIp9vYDWvZebEqpreqGLz4UvmJ4&export=download&confirm=yes"
else:
    print("✅ piper_cache.tgz already present.")
!tar zxvf "{PIPER_TRAINING}/piper_cache.tgz"

# 2. Last Checkpoint
if not os.path.exists(f"{PIPER_TRAINING}/last.ckpt"):
    print("Fetching last.ckpt...")
    !wget -O "{PIPER_TRAINING}/last.ckpt" "https://drive.usercontent.google.com/download?id=1inHU_oii-A1fp99VvLgBJt4vndRKabl4&export=download&confirm=yes"
else:
    print("✅ last.ckpt already present.")

# 3. WAV Checkpoint
if not os.path.exists(f"{PIPER_TRAINING}/en-gb_pisi-suttaplayer-medium-dataset.tgz"):
    print("Fetching en-gb_pisi-suttaplayer-medium-dataset.tgz...")
    !wget -O "{PIPER_TRAINING}/en-gb_pisi-suttaplayer-medium-dataset.tgz" "https://drive.usercontent.google.com/download?id=1xmxElYOtABIWdBFtb5FTVRx1WTe17Bw8&export=download&confirm=yes"
else:
    print("✅ en-gb_pisi-suttaplayer-medium-dataset.tgz already present.")
!tar zxvf "{PIPER_TRAINING}/en-gb_pisi-suttaplayer-medium-dataset.tgz" -C "{PIPER_TRAINING}"


In [5]:
# REPORT CHECKPOINT EPOCH

import torch
import os

# Path to the last.ckpt file
# ckpt_path_to_check = f"{PIPER_TRAINING}/checkpoints/last.ckpt"  # next
ckpt_path_to_check = f"{PIPER_TRAINING}/last.ckpt"  # prev

# Load the checkpoint metadata (without loading the full model weights to save memory)
# state_dict usually contains 'epoch', 'global_step', etc.
checkpoint = torch.load(ckpt_path_to_check, map_location='cpu')

# Check if it has the 'epoch' key (standard in Lightning)
if isinstance(checkpoint, dict):
    epoch = checkpoint.get('epoch', 'Unknown')
    global_step = checkpoint.get('global_step', 'Unknown')
    print(f"✅ Epoch: {epoch}")
    print(f"✅ Global Step: {global_step}")
else:
    print("⚠️  Checkpoint format unexpected. Full state dict loaded.")

✅ Epoch: 9120
✅ Global Step: 20202


In [4]:
from pathlib import Path
import shutil

# ==== CHANGE THESE ====
VOICE_NAME      = "en_gb-suttaplayer-medium"
ESPEAK_VOICE    = "en-gb"  # run `!espeak-ng --voices` to see options
SAMPLE_RATE_HZ  = 22050
BATCH_SIZE      = 8       # drop to 8 or 4 if you OOM

DATA_ROOT       = Path(PIPER_TRAINING)
AUDIO_DIR       = DATA_ROOT / "wavs"
CSV_PATH        = Path(REPO_DIR) / f"corpus-preperation/metadata-phonemes.csv"
PHONEME_MAP_PATH= Path(REPO_DIR) / f"config/en[gb]_pi[si]-suttaplayer-phoneme-map.json"
# TR_CB_SRC_PATH  = Path(f"{REPO_SCRIPTS_DIR}/train_sutta_voice.py")
# TR_CB_DES_PATH  = Path(f"{PIPER_REPO}/train_sutta_voice.py")
# shutil.copy2(TR_CB_SRC_PATH, TR_CB_DES_PATH)

CACHE_DIR       = Path("/content/piper_cache")
CONFIG_PATH     = DATA_ROOT / f"{VOICE_NAME}.json"

# Optional: start from an existing checkpoint to speed up & stabilize training
# Get a .ckpt from https://huggingface.co/datasets/rhasspy/piper-checkpoints (medium quality recommended)
CKPT_PATH       = DATA_ROOT / "last.ckpt"  # e.g., "/content/drive/MyDrive/piper_training/en_GB-northern_english_male-medium.ckpt"

# Make sure dirs exist
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("CSV_PATH exists:", CSV_PATH.exists())
print("PHONEME_MAP_PATH exists:", PHONEME_MAP_PATH.exists())
print("AUDIO_DIR exists:", AUDIO_DIR.exists())

print("CKPT_PATH exists:", CKPT_PATH.exists())
# print("TR_CB_DES_PATH exists:", TR_CB_DES_PATH.exists())
print("CACHE_DIR:", CACHE_DIR)
print("Config will be written to:", CONFIG_PATH)

CSV_PATH exists: True
PHONEME_MAP_PATH exists: True
AUDIO_DIR exists: True
CKPT_PATH exists: True
CACHE_DIR: /content/piper_cache
Config will be written to: /content/drive/MyDrive/piper_training/en_gb-suttaplayer-medium.json


In [8]:
import torch, platform, sys

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi

Python: 3.11.13
PyTorch: 2.6.0+cu124
CUDA available: True
Sun Sep  6 06:53:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             15W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |              

In [5]:
!pip install setuptools==81.0.0
!pip install torch==2.3.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121 --force-reinstall
!pip install torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121
!pip install onnx==1.15.0
!pip install lightning==2.3.3

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.3.1%2Bcu121-cp311-cp311-linux_x86_64.whl (781.0 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.3.1%2Bcu121-cp311-cp311-linux_x86_64.whl (3.4 MB)
  Using cached filelock-3.32.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 176.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 182.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 141.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
print(CSV_PATH)

/content/drive/MyDrive/sutta-tts-model-training/corpus-preperation/metadata-phonemes.csv


In [7]:
!sudo apt-get update -y
!sudo apt-get install -y build-essential cmake ninja-build ffmpeg

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [113 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,920 kB]
Get:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://

In [8]:
%cd /content
!rm -rf piper1-gpl
!git clone https://github.com/OHF-voice/piper1-gpl.git
%cd piper1-gpl
!pwd

/content
Cloning into 'piper1-gpl'...
remote: Enumerating objects: 1761, done.
remote: Counting objects: 100% (421/421), done.
remote: Compressing objects: 100% (244/244), done.
remote: Total 1761 (delta 288), reused 177 (delta 177), pack-reused 1340 (from 3)
Receiving objects: 100% (1761/1761), 23.89 MiB | 19.57 MiB/s, done.
Resolving deltas: 100% (995/995), done.
/content/piper1-gpl
/content/piper1-gpl


In [9]:
!python3 -m pip install -e ".[train]"

Obtaining file:///content/piper1-gpl
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.2/800.2 kB 77.6 MB/s eta 0:00:00
  Building editable for piper-tts (pyproject.toml) ... done
  Created wheel for piper-tts: filename=piper_tts-1.8.0-0.editable-py3-none-any.whl size=18882 sha256=1e8fa2bf664ffa3dce73f3b1d2f79e769f1f2e1bf63e9f6fbf16bcc8cf511c6d
  Stored in directory: /tmp/pip-ephem-wheel-cache-3dug19u1/wheels/c2/10/f7/7fcd14ac900d17ffb0464389baacde317cf11699146f759ddb
Succ

In [10]:
%cd /content/piper1-gpl
!chmod +x ./build_monotonic_align.sh
!./build_monotonic_align.sh

/content/piper1-gpl
Compiling /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx because it changed.
[1/1] Cythonizing /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx
/usr/local/lib/python3.11/dist-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str' for now (Py3). This has changed from earlier releases! File: /content/piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)
performance hint: core.pyx:7:5: Exception check on 'maximum_path_each' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_each' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'maximum_path_each' to allow an error code to be returned.
performance hint: core.pyx:38:6: Exception check on 'maximum_path_c' will always require the GIL to be acquired.
P

In [11]:
!python3 -m pip install --upgrade pip wheel scikit-build cmake ninja

  Using cached wheel-0.48.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached scikit_build-0.19.1-py3-none-any.whl.metadata (21 kB)
  Using cached cmake-4.4.3-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (7.0 kB)
  Using cached ninja-1.13.2-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.7 MB/s eta 0:00:00
Using cached wheel-0.48.0-py3-none-any.whl (33 kB)
Using cached scikit_build-0.19.1-py3-none-any.whl (86 kB)
Using cached cmake-4.4.3-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (30.2 MB)
Using cached ninja-1.13.2-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (183 kB)
  Attempting uninstall: wheel
    Found existing installation: wheel 0.45.1
    Uninstalling wheel-0.45.1:
      Successfully uninstalled wheel-0.45.1
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  A

In [13]:
%cd /content/piper1-gpl
!python3 setup.py build_ext --inplace -v

/content/piper1-gpl


--------------------------------------------------------------------------------
-- Trying 'Ninja' generator
--------------------------------
---------------------------
----------------------
-----------------
------------
-------
--
The --no-warn-unused-cli option is deprecated.  Use -Wno-unused-cli instead.
-- The C compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- The CXX compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Configuring done (1.0s)
-- Generating done (0.0s)
-- Build files have been written to: /content/piper1-gpl/_cmake_test_compile/build
--
-------
------------
-

In [14]:
import pandas as pd, io, os, textwrap

csv_path = str(CSV_PATH)
if os.path.exists(csv_path):
    # Read as pipe-delimited, two columns
    try:
        df = pd.read_csv(csv_path, sep="|", header=None, names=["audio","text"])
        print(df.head())
        # Check a few audio files exist
        missing = [a for a in df["audio"].head(5) if not (AUDIO_DIR/str(a)).exists()]
        print("Missing among first 5:", missing)
    except Exception as e:
        print("CSV read error:", e)
else:
    print("CSV not found at:", csv_path)

   audio                                               text
0  0.wav  ˈɐŋɡəs, mˈɐɡədʰəns, kˈɐsis, kˈosələns, wˈɐdʒdʒ...
1  1.wav  kˈuɹus, pˈɐɲtʃaːləs, mˈɐtʃtʃhəs, sˈuɹəsˌenəs, ...
2  2.wav  pˈuːɹəɳə kˈɐssəpə, mˈɐkkʰəli ɡˈosaːlə, ˈɐdʒitə...
3  3.wav  ðˈə sˈeɪdʒɪz ˈɒv ðˈə pˈast—ˈɐʈʈʰəkə, wˈaːməkə,...
4  4.wav  ˈaɪ hˈav hˈɜːd ðˈat ˈɒn wˈɒn əkˈeɪʒən ðˈə blˈɛ...
Missing among first 5: []


In [12]:
command = f"""
PYTHONPATH={REPO_SCRIPTS_DIR}:$PYTHONPATH \
python3 -m piper.train fit \
  --data.voice_name "{VOICE_NAME}" \
  --data.csv_path "{str(CSV_PATH)}" \
  --data.phoneme_type text \
  --data.phonemes_path "{str(PHONEME_MAP_PATH)}" \
  --data.audio_dir "{str(AUDIO_DIR)}" \
  --model.sample_rate {SAMPLE_RATE_HZ} \
  --data.espeak_voice "{ESPEAK_VOICE}" \
  --data.cache_dir "{str(CACHE_DIR)}" \
  --data.config_path "{str(CONFIG_PATH)}" \
  --data.batch_size {BATCH_SIZE} \
  --trainer.callbacks.class_path "train_sutta_voice.SuttaVoiceUatCallback" \
  --ckpt_path "{CKPT_PATH}" \
  --trainer.accelerator gpu --trainer.devices 1 --trainer.precision 16-mixed \
  --model.mel_fmin 0 \
  --model.mel_fmax 8000
"""
print(command)


PYTHONPATH=/content/drive/MyDrive/sutta-tts-model-training/scripts:$PYTHONPATH python3 -m piper.train fit   --data.voice_name "en_gb-suttaplayer-medium"   --data.csv_path "/content/drive/MyDrive/sutta-tts-model-training/corpus-preperation/metadata-phonemes.csv"   --data.phoneme_type text   --data.phonemes_path "/content/drive/MyDrive/sutta-tts-model-training/config/en[gb]_pi[si]-suttaplayer-phoneme-map.json"   --data.audio_dir "/content/drive/MyDrive/piper_training/wavs"   --model.sample_rate 22050   --data.espeak_voice "en-gb"   --data.cache_dir "/content/piper_cache"   --data.config_path "/content/drive/MyDrive/piper_training/en_gb-suttaplayer-medium.json"   --data.batch_size 8   --trainer.callbacks.class_path "train_sutta_voice.SuttaVoiceUatCallback"   --ckpt_path "/content/drive/MyDrive/piper_training/last.ckpt"   --trainer.accelerator gpu --trainer.devices 1 --trainer.precision 16-mixed   --model.mel_fmin 0   --model.mel_fmax 8000



run the following in terminal

```bash
deno --allow-all /content/drive/MyDrive/sutta-tts-model-training/scripts/training-prune-checkpoints.ts
```


In [ ]:
# ============================================================
# KEEP-ALIVE (Sheets Sync)
# ============================================================
# Run this cell to keep the Colab session alive and sync metrics.

import os
import time
import pandas as pd
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default

# Paths
METRICS_CSV = f"{PIPER_TRAINING}/uat_metrics.csv"
sheet_name = "SuttaPlayer_UAT_Convergence"

try:
    creds, _ = default()
    gc = gspread.authorize(creds)
    try:
        sh = gc.open(sheet_name)
    except:
        sh = gc.create(sheet_name)
        print(f"✅ Created Sheet: {sheet_name}")

    ws = sh.get_worksheet(0)
    last_row = len(ws.col_values(1))

    print(f"💾 Keep-Alive Active. Monitoring CSV...")
    while True:
        # Monitor CSV
        if os.path.exists(METRICS_CSV):
            try:
                df = pd.read_csv(METRICS_CSV)
                if len(df) > last_row:
                    new_data = df.iloc[last_row:].values.tolist()
                    for row in new_data:
                        ws.append_row(row)
                        print(f"  [Sheet] Logged Epoch {row[1]}")
                    last_row = len(df)
            except:
                pass # Ignore write conflicts

        time.sleep(60)
except KeyboardInterrupt:
    print("\n⏹️  Sync Stopped.")

✅ Created Sheet: SuttaPlayer_UAT_Convergence
💾 Keep-Alive Active. Monitoring CSV...
  [Sheet] Logged Epoch 9120
  [Sheet] Logged Epoch 9120
  [Sheet] Logged Epoch 9121
  [Sheet] Logged Epoch 9122
  [Sheet] Logged Epoch 9123
  [Sheet] Logged Epoch 9124
  [Sheet] Logged Epoch 9125
  [Sheet] Logged Epoch 9126
  [Sheet] Logged Epoch 9127
  [Sheet] Logged Epoch 9128
  [Sheet] Logged Epoch 9129


1. Press F12 (or right-click and select Inspect) inside Brave/Chrome to open the Developer Tools.
2. Click on the Console tab.
3. Paste the following JavaScript code and press Enter:

```js
function KeepAlive() {
  let connectBtn = document.querySelector("#connect") || document.querySelector("colab-connect-button");
  if (connectBtn) {
    console.log("Simulating click on Connect Button...");
    connectBtn.click();
  }
}
setInterval(KeepAlive, 60000); // Triggers every 60 seconds
```
